# Embeddings and Vector Stores

Text chunks (**08. Documents and Text Splitting**) must become **vectors** before similarity search can find them. **Embedding models** map text to dense float arrays; **vector stores** persist those vectors with document text and metadata for fast nearest-neighbor lookup.

This notebook covers **`OpenAIEmbeddings`**, the **`Embeddings`** interface, indexing with **`Chroma`** via **`langchain-chroma`**, `similarity_search`, scored search, metadata filters, adding and updating documents, batch embedding, and preparing an indexed collection for **10. Retrievers** and **11. RAG with LCEL**.

You need an OpenAI API key and packages: `langchain-openai`, `langchain-chroma`, `chromadb`.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
import uuid

import langchain_core
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

print("langchain-core:", langchain_core.__version__)

---

## 1. Embeddings → Vector Store → Retrieval

```
Documents (text + metadata)
        │
        ▼
   Embeddings model     page_content → vector[float]
        │
        ▼
   Vector store         store (vector, text, metadata)
        │
        ▼
   Query embedding      user question → vector[float]
        │
        ▼
   Similarity search      top-k nearest vectors → Documents
```

LangChain separates **embedding** (model) from **storage** (vector DB). Swap Chroma for Pinecone or pgvector by changing the vector store class — embeddings and retriever patterns stay similar.

---

## 2. The Embeddings Interface

All embedding integrations implement **`Embeddings`** with two core methods:

| Method | Input | Use |
|--------|-------|-----|
| **`embed_documents(texts)`** | `list[str]` | Index many chunks (batch API call) |
| **`embed_query(text)`** | `str` | Embed a single search query |

Some providers treat queries and documents differently internally; LangChain exposes both methods so integrations can optimize each case.

---

## 3. OpenAIEmbeddings

**`OpenAIEmbeddings`** from `langchain-openai` wraps OpenAI embedding APIs. Default model for this track: **`text-embedding-3-small`** (cost-effective, strong quality).

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY,
)

query_vec = embeddings.embed_query("JWT authentication header")
doc_vecs = embeddings.embed_documents([
    "JWT bearer tokens use Authorization header.",
    "Alembic applies database migrations.",
])

print("query vector dim:", len(query_vec))
print("doc vectors:", len(doc_vecs), "x", len(doc_vecs[0]))
print("first 5 dims:", [round(v, 4) for v in query_vec[:5]])

### 3.1 Cosine Similarity Intuition

Similar meaning → vectors point in similar directions → **high cosine similarity**:

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

In [ ]:
def cosine_sim(a: list[float], b: list[float]) -> float:
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))


pairs = [
    ("JWT auth header", embeddings.embed_query("JWT auth header")),
    ("Bearer token", embeddings.embed_query("Bearer token")),
    ("Alembic migration", embeddings.embed_query("Alembic migration")),
]

jwt_vec = pairs[0][1]
print("Similarity to 'JWT auth header':")
for label, vec in pairs:
    print(f"  {label:20s} {cosine_sim(jwt_vec, vec):.4f}")

---

## 4. Lesson Corpus — Documents to Index

We build the same teaching corpus used in **08. Documents and Text Splitting** (standalone — no file dependency):

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

RAW_CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE.", "source": "api-docs"},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions.", "source": "db-docs"},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header.", "source": "security-docs"},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines.", "source": "test-docs"},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files.", "source": "db-docs"},
]

base_docs = [
    Document(page_content=d["text"], metadata={"id": d["id"], "source": d["source"]})
    for d in RAW_CORPUS
]

LONG_GUIDE = """# Notes API Overview
The Notes API is built with FastAPI.
## Authentication
JWT bearer tokens authenticate API requests.
## Migrations
Alembic applies SQLAlchemy schema migrations.
## Testing
Pytest fixtures share database setup in conftest.py.
"""

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
guide_chunks = splitter.split_documents([
    Document(page_content=LONG_GUIDE, metadata={"source": "guide.md", "id": "guide"})
])

CORPUS: list[Document] = base_docs + guide_chunks
print(f"CORPUS: {len(CORPUS)} documents ({len(base_docs)} base + {len(guide_chunks)} from guide)")

---

## 5. Chroma Vector Store

**Chroma** is an open-source embedding database — local, lightweight, ideal for learning and prototypes. **`langchain-chroma`** provides the LangChain **`Chroma`** vector store class.

| Feature | Detail |
|---------|--------|
| **Deployment** | In-memory or persistent directory |
| **Similarity** | Configurable (cosine default in LangChain usage) |
| **Metadata** | Filter on ingest and query |

In [ ]:
from langchain_chroma import Chroma

COLLECTION = f"langchain_lesson_{uuid.uuid4().hex[:8]}"

vectorstore = Chroma.from_documents(
    documents=CORPUS,
    embedding=embeddings,
    collection_name=COLLECTION,
)

print("collection:", COLLECTION)
print("indexed count:", vectorstore._collection.count())

### 5.1 from_documents — What Happens

1. Extract `page_content` from each **`Document`**
2. Call **`embeddings.embed_documents(texts)`** (batched)
3. Store vectors + text + metadata in Chroma collection

One call indexes the full corpus — the standard ingest path for RAG prototypes.

---

## 6. similarity_search

Embed the query, find **top-k** nearest vectors, return **`Document`** objects:

In [ ]:
def show_hits(question: str, k: int = 3) -> None:
    hits = vectorstore.similarity_search(question, k=k)
    print(f"Q: {question}\n")
    for i, doc in enumerate(hits, 1):
        mid = doc.metadata.get("id", "?")
        src = doc.metadata.get("source", "?")
        print(f"  {i}. [{mid}] ({src}) {doc.page_content[:70]}...")
    print()


show_hits("How do clients authenticate to the API?")
show_hits("What command applies Alembic migrations?")
show_hits("pytest fixtures and conftest.py")

---

## 7. similarity_search_with_score

Returns **`(Document, score)`** tuples. Chroma returns **distance** (lower = more similar for L2; for cosine space, interpret per Chroma docs).

In [ ]:
scored = vectorstore.similarity_search_with_score("JWT bearer token header", k=4)

print(f"{'score':>8s}  {'id':6s}  content")
print("-" * 70)
for doc, score in scored:
    print(f"{score:8.4f}  {doc.metadata.get('id', '?'):6s}  {doc.page_content[:50]}...")

Use scores for **thresholding** (reject weak matches) and debugging retrieval quality — expanded in **10. Retrievers**.

---

## 8. Metadata Filtering

Restrict search to documents matching metadata — e.g. only **`security-docs`**:

In [ ]:
filtered = vectorstore.similarity_search(
    "authentication tokens",
    k=3,
    filter={"source": "security-docs"},
)

print("filter source=security-docs:")
for d in filtered:
    print(f"  [{d.metadata.get('id')}] {d.page_content}")

db_only = vectorstore.similarity_search(
    "schema migrations",
    k=3,
    filter={"source": "db-docs"},
)
print("\nfilter source=db-docs:")
for d in db_only:
    print(f"  [{d.metadata.get('id')}] {d.page_content[:60]}...")

Metadata filters enforce **ACLs** (user sees only their tenant's docs) and **domain routing** (search only HR policies).

---

## 9. Adding and Updating Documents

Index new chunks without rebuilding the entire store:

In [ ]:
new_doc = Document(
    page_content="OpenAPI docs are served at /docs and /redoc in development.",
    metadata={"id": "c6", "source": "api-docs"},
)

added_ids = vectorstore.add_documents([new_doc])
print("added ids:", added_ids)
print("count after add:", vectorstore._collection.count())

hit = vectorstore.similarity_search("Where is Swagger UI?", k=1)
print("retrieve new doc:", hit[0].page_content)

### 9.1 delete — Remove by ID

Remove stale chunks when source documents change:

In [ ]:
if added_ids:
    vectorstore.delete(ids=added_ids)
    print("deleted c6, count:", vectorstore._collection.count())

Production pipelines version collections or run incremental upserts — patterns in **17. Production Patterns for LangChain**.

---

## 10. Persistent vs In-Memory Chroma

| Mode | Constructor | Use |
|------|-------------|-----|
| **In-memory** | `Chroma.from_documents(...)` (default) | Notebooks, tests |
| **Persistent** | `persist_directory="./chroma_data"` | Local dev restart survival |

```python
# Persistent example (not run here to avoid disk clutter)
vectorstore = Chroma.from_documents(
    documents=CORPUS,
    embedding=embeddings,
    collection_name="notes_api",
    persist_directory="./chroma_notes_api",
)
```

Reload an existing store:

```python
vectorstore = Chroma(
    collection_name="notes_api",
    embedding_function=embeddings,
    persist_directory="./chroma_notes_api",
)
```

---

## 11. Embeddings as Runnable

LangChain wraps embeddings for LCEL with **`embeddings.embed_query`** passthrough or dedicated runnable helpers. Vector stores call embeddings internally — you rarely pipe embeddings directly in RAG chains.

In [ ]:
from langchain_core.runnables import RunnableLambda

embed_query_runnable = RunnableLambda(lambda q: embeddings.embed_query(q))
vec = embed_query_runnable.invoke("What is FastAPI?")
print("runnable embed dim:", len(vec))

---

## 12. Batch Embedding and Cost

OpenAI bills embedding calls by **token**. Batch **`embed_documents`** in one API request when indexing large corpora.

In [ ]:
texts = [d.page_content for d in CORPUS]
char_counts = [len(t) for t in texts]

print(f"chunks: {len(texts)}")
print(f"total chars: {sum(char_counts)}")
print(f"avg chars/chunk: {np.mean(char_counts):.1f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(char_counts)), char_counts, color="#4c72b0")
ax.set_xlabel("chunk index")
ax.set_ylabel("characters")
ax.set_title("Corpus chunk sizes (embedding cost correlates with tokens)")
plt.tight_layout()
plt.show()

| Model | Dimensions | Typical use |
|-------|------------|-------------|
| **`text-embedding-3-small`** | 1536 | Default; cost-efficient |
| **`text-embedding-3-large`** | 3072 | Higher quality, higher cost |

Optional: pass **`dimensions=N`** to `OpenAIEmbeddings` for Matryoshka-style reduced dimensions (OpenAI 3.x models).

---

## 13. max_marginal_relevance_search (Preview)

Plain similarity search may return **redundant** chunks. **MMR** balances relevance with diversity — configured via **`.as_retriever(search_type="mmr")`** in **10. Retrievers**.

In [ ]:
mmr_hits = vectorstore.max_marginal_relevance_search(
    "Alembic migrations and schema",
    k=3,
    fetch_k=6,
    lambda_mult=0.5,
)

print("MMR results (diverse):")
for d in mmr_hits:
    print(f"  [{d.metadata.get('id', '?')}] {d.page_content[:55]}...")

---

## 14. Vector Store → Retriever Bridge

Vector stores expose **`.as_retriever()`** — a Runnable that returns `list[Document]` for a query string. Full retriever options in **10. Retrievers**.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
docs = retriever.invoke("JWT Authorization header")
print("retriever returned", len(docs), "documents")
for d in docs:
    print(f"  [{d.metadata.get('id')}] {d.page_content[:60]}...")

---

## 15. Ingest Checklist

| Step | Action |
|------|--------|
| 1 | Load / build **`Document`** list (**08**) |
| 2 | Split long sources |
| 3 | Enrich metadata (`id`, `source`, `chunk_index`) |
| 4 | Choose **`OpenAIEmbeddings`** model |
| 5 | **`Chroma.from_documents`** or batch add |
| 6 | Smoke-test **`similarity_search`** |
| 7 | Expose **`.as_retriever()`** in RAG chain (**11**) |

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Different embed model at query vs index | Poor recall | Same `embeddings` instance / model name |
| No metadata on chunks | Cannot filter or cite | Use `Document.metadata` |
| Huge unsplit documents | One vector for entire book | Split first (**08**) |
| Ignoring similarity scores | Bad chunks in context | Threshold or MMR (**10**) |
| Re-embedding entire corpus hourly | Cost spike | Incremental `add_documents` / delete |
| In-memory store in production | Data loss on restart | `persist_directory` or hosted DB |

---

## 17. Summary

**Embeddings** convert text to vectors; **vector stores** persist vectors with documents for similarity search. **`OpenAIEmbeddings`** handles API calls; **`Chroma.from_documents`** indexes a corpus in one step.

Key takeaways:

- Use **`embed_documents`** for indexing, **`embed_query`** for search.
- **`similarity_search`** and **`similarity_search_with_score`** return **`Document`** hits.
- **Metadata filters** scope search to sources or tenants.
- **`add_documents`** / **`delete`** support incremental updates.
- **`.as_retriever()`** bridges to LCEL RAG chains (**10**, **11**).

Demonstrations indexed the lesson corpus, ran semantic search, filtered by metadata, added/deleted documents, visualized chunk sizes, previewed MMR, and wired a retriever.

Next: **10. Retrievers** — `as_retriever` options, MMR, thresholds, and custom retriever runnables.